# Hyperparameter Sweep — Latent JacobianODE

This notebook launches a hyperparameter sweep for the **Latent JacobianODE** model.
It supports two training modes:

- **`from_scratch`**: Encoder and JacobianODE are trained jointly from random initialization.
- **`pretrained`**: A pre-trained encoder (loaded from W&B) is used; only the JacobianODE is trained (encoder optionally fine-tuned).

**Workflow:**
1. Select the training mode and configure hyperparameters (Sections 1–3)
2. Build the Hydra config and generate data (Sections 4–5)
3. Launch the Hydra `--multirun` sweep via SLURM (Section 6)
4. After training completes, use the companion **Sweep Analytics** notebook for model selection and diagnostics.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import itertools
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf
import os
import subprocess
import torch
from tqdm.auto import tqdm
import wandb

from JacobianODE.jacobians import (
    load_config,
    initialize_config,
    seed_everything,
    make_trajectories,
    postprocess_data,
    create_dataloaders,
)
from JacobianODE.jacobians.tuning import DEFAULT_LAMBDA_LOOP_VALUES

torch.set_float32_matmul_precision('high')

## 1. Training Mode

Set `MODE` to `"from_scratch"` or `"pretrained"`.

- **`from_scratch`**: Define encoder architecture below (Section 2a).
- **`pretrained`**: Specify the W&B project and run ID of the pre-trained encoder (Section 2b).

In [ ]:
# ----------------------------------------------------------------
# Training mode: "from_scratch" or "pretrained"
# ----------------------------------------------------------------
MODE = "from_scratch"  # <-- CHANGE THIS

assert MODE in ("from_scratch", "pretrained"), f"Invalid MODE: {MODE}"
print(f"Training mode: {MODE}")

In [ ]:
# ----------------------------------------------------------------
# Paths and W&B settings
# ----------------------------------------------------------------
SAVE_DIR     = "/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs"
WANDB_ENTITY = "JacobianODE" 

## 2a. From-Scratch Encoder Settings

*Skip this section if `MODE = "pretrained"`.*

In [ ]:
# ================================================================
# From-scratch encoder settings (skip if MODE == "pretrained")
# ================================================================
if MODE == "from_scratch":
    # MLP encoder (pointwise: same MLP at every timestep)
    MLP_HIDDEN_DIM = 256
    MLP_N_LAYERS   = 3
    MLP_DROPOUT    = 0.1

    # Decoder MLP (pointwise: z_t -> x_hat_t)
    DECODER_HIDDEN = 128
    DECODER_LAYERS = 2

    ENCODER_WARMUP_EPOCHS = 5

## 2b. Pre-trained Encoder Settings

*Skip this section if `MODE = "from_scratch"`.*

In [ ]:
# ================================================================
# Pre-trained encoder settings (skip if MODE == "from_scratch")
# ================================================================
if MODE == "pretrained":
    from JacobianODE.encoder_only.pretrained import load_pretrained_encoder

    ENCODER_SAVE_DIR = "/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/encoder_runs"

    # Encoder source on W&B
    ENCODER_PROJECT = "Lorenz_MLPFULL_Normed_L6__EncoderOnly"
    ENCODER_RUN_ID  = "vc98usdl"

    FREEZE_ENCODER = False

    adapter, encoder_cfg, encoder_run = load_pretrained_encoder(
        project=ENCODER_PROJECT,
        run_id=ENCODER_RUN_ID,
        save_dir=ENCODER_SAVE_DIR,
        freeze=FREEZE_ENCODER,
        verbose=True,
    )

    print(f"Encoder type:    {type(adapter.encoder).__name__}")
    print(f"n_latent:        {adapter.n_latent}")
    print(f"context_margin:  {adapter.context_margin}")
    print(f"Frozen:          {FREEZE_ENCODER}")

## 3. Hyperparameters

In [ ]:
# ----------------------------------------------------------------
# Data hyperparameters
# ----------------------------------------------------------------
if MODE == "pretrained":
    # Inherit data settings from the pre-trained encoder config
    NUM_ICS          = int(encoder_cfg.data.trajectory_params.num_ics)
    N_PERIODS        = int(encoder_cfg.data.trajectory_params.n_periods)
    PTS_PER_PERIOD   = int(encoder_cfg.data.trajectory_params.pts_per_period)
    OBS_NOISE        = float(encoder_cfg.data.postprocessing.obs_noise)
    NORMALIZE        = encoder_cfg.data.postprocessing.normalize
    delay_params     = encoder_cfg.data.train_test_params.delay_embedding_params
    OBSERVED_INDICES = list(delay_params.observed_indices) if delay_params.observed_indices != "all" else "all"
    N_DELAYS         = int(delay_params.get("n_delays", 1))
    DELAY_SPACING    = int(delay_params.get("delay_spacing", 1))
else:
    OBS_NOISE        = 0.01
    OBSERVED_INDICES = [0, 1, 2]
    N_PERIODS        = 12
    PTS_PER_PERIOD   = 100
    NUM_ICS          = 32
    N_DELAYS         = 1
    DELAY_SPACING    = 1
    NORMALIZE        = True

# Known Lorenz Lyapunov exponents (sigma=10, rho=28, beta=8/3)
TRUE_LYAPUNOV = [0.91, 0.0, -14.57]

print(f"Data: NUM_ICS={NUM_ICS}, N_PERIODS={N_PERIODS}, PTS_PER_PERIOD={PTS_PER_PERIOD}")
print(f"  OBS_NOISE={OBS_NOISE}, NORMALIZE={NORMALIZE}")
print(f"  OBSERVED_INDICES={OBSERVED_INDICES}, N_DELAYS={N_DELAYS}, DELAY_SPACING={DELAY_SPACING}")

In [ ]:
# ----------------------------------------------------------------
# Latent space
# ----------------------------------------------------------------
if MODE == "pretrained":
    N_LATENT = adapter.n_latent
else:
    N_LATENT = 3

# ----------------------------------------------------------------
# Jacobian MLP
# ----------------------------------------------------------------
JAC_HIDDEN_DIM = [256, 1024, 2048, 2048]
JAC_NUM_LAYERS = 4
JAC_ACTIVATION = 'silu'

print(f"N_LATENT: {N_LATENT}")
print(f"Jacobian MLP: {JAC_NUM_LAYERS} layers, hidden_dim={JAC_HIDDEN_DIM}")

In [ ]:
# ----------------------------------------------------------------
# Training hyperparameters
# ----------------------------------------------------------------
BATCH_SIZE               = 32 if MODE == "from_scratch" else 16
MAX_EPOCHS               = 150 if MODE == "from_scratch" else 200
LIMIT_TRAIN_BATCHES      = 200
LIMIT_VAL_BATCHES        = 50
ACCUMULATE_GRAD_BATCHES  = 1 if MODE == "from_scratch" else 4
LEARNING_RATE            = 1e-4
WEIGHT_DECAY             = 1e-4

TRAJ_INIT_STEPS          = 15
PREDICTION_STEPS         = 30
INTERP_PTS               = 4
INNER_N                  = 20
JAC_WINDOW_STRIDE        = 5 if MODE == "from_scratch" else PREDICTION_STEPS

RECONSTRUCTION_LOSS_WEIGHT    = 1.0 if MODE == "from_scratch" else (0.0 if (MODE == "pretrained" and FREEZE_ENCODER) else 1.0)
LATENT_PREDICTION_LOSS_WEIGHT = 1.0
JAC_CONSISTENCY_WEIGHT        = 0.0

# Noise injection (pretrained only)
OBS_NOISE_SCALE   = 0.0
LATENT_NOISE_SCALE = 0.0 if MODE == "from_scratch" else 0.05
LATENT_NOISE_PER_STEP = True
PRECOMPUTE_LATENT_NOISE_FACTOR = True

# Homoscedastic uncertainty weighting (from_scratch only)
LEARN_R2_WEIGHT          = False
LEARN_LOOP_CLOSURE_WEIGHT = False
LEARN_FNN_WEIGHT         = False
LEARN_JAC_CONS_WEIGHT    = False
LEARN_JAC_NORM_WEIGHT    = False
LOG_VAR_INIT             = 'auto'

EARLY_STOPPING_PATIENCE       = 2
EARLY_STOPPING_MIN_EPOCHS     = 10
EARLY_STOPPING_PERCENT_THRESH = 0.01

In [ ]:
# ----------------------------------------------------------------
# Sweep parameters
# ----------------------------------------------------------------
LAMBDA_LOOP_CLOSURE_VALUES = [0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10]
LAMBDA_FNN_VALUES = [0]

SWEEP_PARAMS = {
    "training.lightning.loop_closure_weight": LAMBDA_LOOP_CLOSURE_VALUES,
    "training.lightning.fnn_weight": LAMBDA_FNN_VALUES,
    "training.lightning.fnn_normalize": [True],
    "training.lightning.fnn_elementwise_regularization": [True],
}

# Build all parameter combinations
sweep_keys = list(SWEEP_PARAMS.keys())
sweep_value_lists = [SWEEP_PARAMS[k] for k in sweep_keys]
all_sweep_combos = [
    dict(zip(sweep_keys, vals))
    for vals in itertools.product(*sweep_value_lists)
]
n_sweep_combos = len(all_sweep_combos)

# For local config: use first combo's values
_first_combo = all_sweep_combos[0]
FNN_WEIGHT_FOR_CONFIG = _first_combo.get("training.lightning.fnn_weight", 0)
LAMBDA_LOOP_FOR_CONFIG = _first_combo.get("training.lightning.loop_closure_weight", 0)

print(f"Sweep grid: {n_sweep_combos} total combinations")
for k, v in SWEEP_PARAMS.items():
    print(f"  {k}: {v}")

In [ ]:
# ----------------------------------------------------------------
# W&B project and group
# ----------------------------------------------------------------
if MODE == "from_scratch":
    _obs_idx_label = ''.join(str(i) for i in OBSERVED_INDICES) if OBSERVED_INDICES != "all" else "all"
    WANDB_PROJECT = f"Lorenz_IND{_obs_idx_label}_N{N_DELAYS}_D{DELAY_SPACING}_Norm{NORMALIZE}_L{N_LATENT}__JacobianODE"
else:
    # Set explicitly for pretrained runs, or auto-generate
    WANDB_PROJECT = None  # Set to a string to override; None = auto-generate below
    if WANDB_PROJECT is None:
        WANDB_PROJECT = f"Lorenz_Pretrained_L{N_LATENT}__JacobianODE"

WANDB_PROJECT_PATH = f"{WANDB_ENTITY}/{WANDB_PROJECT}"

# Optional group to organize runs within a project
# WANDB_GROUP = None
WANDB_GROUP = f"sweep_{MODE}"

print(f"W&B project: {WANDB_PROJECT_PATH}")
print(f"W&B group:   {WANDB_GROUP}")

In [ ]:
# ----------------------------------------------------------------
# Sequence length calculation
# ----------------------------------------------------------------
JAC_WINDOW = TRAJ_INIT_STEPS + PREDICTION_STEPS
SEQ_LENGTH = JAC_WINDOW

n_sub_windows = (SEQ_LENGTH - JAC_WINDOW) // JAC_WINDOW_STRIDE + 1
print(f"JacobianODE window:      {JAC_WINDOW} = {TRAJ_INIT_STEPS} init + {PREDICTION_STEPS} pred")
print(f"Dataset sequence length: {SEQ_LENGTH}")
print(f"Sub-windows per batch:   {n_sub_windows}")
print(f"Effective samples/step:  {BATCH_SIZE * n_sub_windows}")

## 4. Config Setup

Build the base Hydra config. This config is used for local data generation and as the
template for the sweep command.

In [ ]:
_hidden_dim_str = str(JAC_HIDDEN_DIM).replace(' ', '')
_obs_idx_str = str(list(OBSERVED_INDICES)).replace(' ', '') if OBSERVED_INDICES != "all" else "all"

# ----------------------------------------------------------------
# Shared overrides (both modes)
# ----------------------------------------------------------------
overrides = [
    # Jacobian MLP
    f"model.params.hidden_dim={_hidden_dim_str}",
    f"model.params.num_layers={JAC_NUM_LAYERS}",
    f"model.params.activation={JAC_ACTIVATION}",
    f"model.prediction_steps={PREDICTION_STEPS}",

    # Data
    "data=dysts",
    "data.flow._target_=JacobianODE.dysts_sim.flows.Lorenz",
    f"data.trajectory_params.n_periods={N_PERIODS}",
    f"data.trajectory_params.pts_per_period={PTS_PER_PERIOD}",
    f"data.trajectory_params.num_ics={NUM_ICS}",
    f"data.postprocessing.obs_noise={OBS_NOISE}",
    f"data.postprocessing.normalize={NORMALIZE}",
    f"data.train_test_params.delay_embedding_params.observed_indices={_obs_idx_str}",
    f"data.train_test_params.delay_embedding_params.n_delays={N_DELAYS}",
    f"data.train_test_params.delay_embedding_params.delay_spacing={DELAY_SPACING}",
    f"data.train_test_params.seq_length={SEQ_LENGTH}",

    # Training
    f"training.batch_size={BATCH_SIZE}",
    f"training.logger.save_dir={SAVE_DIR}",
    f"training.lightning.optimizer_kwargs.lr={LEARNING_RATE}",
    f"training.lightning.optimizer_kwargs.weight_decay={WEIGHT_DECAY}",
    "training.lightning.loop_closure_training=True",
    "training.lightning.trajectory_training=True",
    f"training.lightning.reconstruction_loss_weight={RECONSTRUCTION_LOSS_WEIGHT}",
    f"training.lightning.latent_prediction_loss_weight={LATENT_PREDICTION_LOSS_WEIGHT}",
    f"training.lightning.jac_consistency_weight={JAC_CONSISTENCY_WEIGHT}",
    f"training.lightning.fnn_weight={FNN_WEIGHT_FOR_CONFIG}",

    # Teacher forcing
    "training.lightning.alpha_teacher_forcing=1",
    "training.lightning.teacher_forcing_annealing=True",
    "training.lightning.gamma_teacher_forcing=0.999",

    # JacobianODEint
    f"training.lightning.jacobianODEint_kwargs.traj_init_steps={TRAJ_INIT_STEPS}",
    f"training.lightning.jacobianODEint_kwargs.interp_pts={INTERP_PTS}",
    f"training.lightning.jacobianODEint_kwargs.inner_N={INNER_N}",
    "training.lightning.jacobianODEint_kwargs.inner_path=line",

    # Trainer
    f"training.trainer_params.max_epochs={MAX_EPOCHS}",
    f"training.trainer_params.limit_train_batches={LIMIT_TRAIN_BATCHES}",
    f"training.trainer_params.limit_val_batches={LIMIT_VAL_BATCHES}",
    f"training.trainer_params.accumulate_grad_batches={ACCUMULATE_GRAD_BATCHES}",

    # Early stopping
    f"training.early_stopping.early_stopping_patience={EARLY_STOPPING_PATIENCE}",
    f"training.early_stopping.percent_thresh={EARLY_STOPPING_PERCENT_THRESH}",
    f"training.early_stopping.min_epochs={EARLY_STOPPING_MIN_EPOCHS}",
]

# ----------------------------------------------------------------
# Mode-specific overrides
# ----------------------------------------------------------------
if MODE == "from_scratch":
    overrides += [
        "model=latent_mlp",
        f"model.encoder.n_input={len(OBSERVED_INDICES) if OBSERVED_INDICES != 'all' else 3}",
        f"model.encoder.n_latent={N_LATENT}",
        f"model.encoder.hidden_dim={MLP_HIDDEN_DIM}",
        f"model.encoder.n_layers={MLP_N_LAYERS}",
        f"model.encoder.dropout={MLP_DROPOUT}",
        f"model.encoder.decoder_hidden={DECODER_HIDDEN}",
        f"model.encoder.decoder_layers={DECODER_LAYERS}",
        "model.encoder.context_margin=0",
        f"model.jac_window_stride={JAC_WINDOW_STRIDE}",
        f"model.encoder_warmup_epochs={ENCODER_WARMUP_EPOCHS}",

        f"training.lightning.learn_r2_weight={LEARN_R2_WEIGHT}",
        f"training.lightning.learn_loop_closure_weight={LEARN_LOOP_CLOSURE_WEIGHT}",
        f"training.lightning.learn_fnn_weight={LEARN_FNN_WEIGHT}",
        f"training.lightning.learn_jac_cons_weight={LEARN_JAC_CONS_WEIGHT}",
        f"training.lightning.learn_jac_norm_weight={LEARN_JAC_NORM_WEIGHT}",
        f"training.lightning.log_var_init={LOG_VAR_INIT}",

        f"training.logger_save_dirs={SAVE_DIR}",
    ]
    config_name = "config"

elif MODE == "pretrained":
    # Detect encoder architecture for Hydra model config
    _enc_cls = type(adapter.encoder).__name__
    if "MLP" in _enc_cls:
        MODEL_CONFIG = "latent_mlp"
    elif "SSM" in _enc_cls or "LRU" in _enc_cls:
        MODEL_CONFIG = "latent_ssm"
    elif "Transformer" in _enc_cls:
        MODEL_CONFIG = "latent_transformer"
    elif "TCN" in _enc_cls:
        MODEL_CONFIG = "latent_tcn"
    else:
        MODEL_CONFIG = "latent_ssm"

    _enc = encoder_cfg.model.encoder
    _encoder_overrides = [f"model.encoder.n_latent={N_LATENT}"]
    if MODEL_CONFIG == "latent_ssm":
        _encoder_overrides += [
            f"model.encoder.d_model={int(_enc.get('d_model', 64))}",
            f"model.encoder.d_state={int(_enc.get('d_state', 64))}",
            f"model.encoder.n_layers={int(_enc.get('n_layers', 3))}",
            f"model.encoder.ffn_expand={int(_enc.get('ffn_expand', 2))}",
            f"model.encoder.r_min={float(_enc.get('r_min', 0.0))}",
            f"model.encoder.r_max={float(_enc.get('r_max', 0.99))}",
            f"model.encoder.dropout={float(_enc.get('dropout', 0.1))}",
            f"model.encoder.use_positional_encoding={bool(_enc.get('use_positional_encoding', True))}",
            f"model.encoder.positional_encoding_type={_enc.get('positional_encoding_type', 'sinusoidal')}",
            f"model.encoder.decoder_hidden={int(_enc.get('decoder_hidden', 128))}",
            f"model.encoder.decoder_layers={int(_enc.get('decoder_layers', 2))}",
            f"model.encoder.context_margin={adapter.context_margin}",
        ]
    elif MODEL_CONFIG == "latent_transformer":
        _encoder_overrides += [
            f"model.encoder.d_model={int(_enc.get('d_model', 64))}",
            f"model.encoder.n_heads={int(_enc.get('n_heads', 4))}",
            f"model.encoder.n_layers={int(_enc.get('n_layers', 3))}",
            f"model.encoder.dim_feedforward={int(_enc.get('dim_feedforward', 128))}",
            f"model.encoder.dropout={float(_enc.get('dropout', 0.1))}",
            f"model.encoder.decoder_hidden={int(_enc.get('decoder_hidden', 128))}",
            f"model.encoder.decoder_layers={int(_enc.get('decoder_layers', 2))}",
            f"model.encoder.context_margin={adapter.context_margin}",
        ]
    elif MODEL_CONFIG == "latent_tcn":
        _encoder_overrides += [
            f"model.encoder.n_channels={int(_enc.get('n_channels', 64))}",
            f"model.encoder.kernel_size={int(_enc.get('kernel_size', 7))}",
            f"model.encoder.n_layers={int(_enc.get('n_layers', 4))}",
            f"model.encoder.dropout={float(_enc.get('dropout', 0.1))}",
            f"model.encoder.decoder_hidden={int(_enc.get('decoder_hidden', 128))}",
            f"model.encoder.decoder_layers={int(_enc.get('decoder_layers', 2))}",
            f"model.encoder.context_margin={adapter.context_margin}",
        ]

    overrides += [
        f"model={MODEL_CONFIG}",
        *_encoder_overrides,

        # Pretrained encoder source
        f"pretrained_encoder.project={ENCODER_PROJECT}",
        f"pretrained_encoder.run_id={ENCODER_RUN_ID}",
        f"pretrained_encoder.save_dir={ENCODER_SAVE_DIR}",
        f"pretrained_encoder.freeze={FREEZE_ENCODER}",

        # Noise injection
        f"training.lightning.obs_noise_scale={OBS_NOISE_SCALE}",
        f"training.lightning.latent_noise_scale={LATENT_NOISE_SCALE}",
        f"training.lightning.latent_noise_per_step={LATENT_NOISE_PER_STEP}",
        f"training.lightning.precompute_latent_noise_factor={PRECOMPUTE_LATENT_NOISE_FACTOR}",
    ]
    config_name = "pretrained_config"

# ----------------------------------------------------------------
# Build config
# ----------------------------------------------------------------
cfg = load_config(config_name=config_name, overrides=overrides)
cfg = initialize_config(cfg)

# Update WANDB_PROJECT if auto-generating for pretrained mode
if MODE == "pretrained":
    data_cls = cfg.data.flow._target_.split('.')[-1]
    if WANDB_PROJECT.startswith("Lorenz_Pretrained"):
        WANDB_PROJECT = f"{data_cls}_Pretrained_L{N_LATENT}__JacobianODE"
        WANDB_PROJECT_PATH = f"{WANDB_ENTITY}/{WANDB_PROJECT}"

print(f"\nConfig built successfully.")
print(f"  Lightning target: {cfg.training.lightning._target_}")
print(f"  Encoder: n_latent={N_LATENT}")
print(f"  Jacobian MLP: input_dim={cfg.model.params.input_dim}, output_dim={cfg.model.params.output_dim}")
print(f"  W&B project: {WANDB_PROJECT_PATH}")

## 5. Generate Data

Generate trajectories, apply postprocessing (noise + normalization), and build dataloaders.

In [ ]:
seed_everything(cfg.data.flow.random_state)
eq, sol, dt = make_trajectories(cfg, verbose=True)
print(f"\nFull trajectory shape: {sol['values'].shape}")
print(f"Time step dt = {dt:.4f}")

In [ ]:
result = postprocess_data(cfg, sol["values"])
values = result.values
mu = result.mu
sigma = result.sigma
noise_scale_factor = result.noise_scale_factor

cfg.data.postprocessing.mu = float(mu)
cfg.data.postprocessing.sigma = float(sigma)
cfg.data.postprocessing.noise_scale_factor = float(noise_scale_factor)

train_dl, val_dl, test_dl, trajs = create_dataloaders(cfg, values, verbose=True, return_full_obs=True)
n_obs = trajs['train_trajs'].sequence.shape[-1]
n_dims = values.shape[-1]

print(f"\nn_obs = {n_obs}, n_dims = {n_dims}")
print(f"sqrt(n_dims) = {np.sqrt(n_dims):.4f}  (C2 loop-closure threshold)")

## 6. Launch Sweep

Uses Hydra `--multirun` with the SLURM launcher. Checks W&B for already-completed
runs and only launches jobs for remaining parameter combinations.

> **Wait for all SLURM jobs to finish** before running the companion Analytics notebook.

In [ ]:
def _get_sweep_param_from_run(run, key):
    """Extract a sweep parameter value from a W&B run config."""
    parts = key.split('.')
    val = run.config
    for p in parts:
        if not isinstance(val, dict) or p not in val:
            return None
        val = val[p]
    return val


def _combo_matches_run(combo, run):
    """Check if a parameter combination matches a finished W&B run."""
    for key, target_val in combo.items():
        run_val = _get_sweep_param_from_run(run, key)
        if run_val is None:
            return False
        if isinstance(target_val, bool):
            if bool(run_val) != target_val:
                return False
        elif isinstance(target_val, (int, float)):
            if abs(float(run_val) - float(target_val)) > 1e-10:
                return False
        else:
            if str(run_val) != str(target_val):
                return False
    return True


def _is_jac_ode_run(run):
    """Return True if this run has an encoder (i.e. is a JacobianODE run)."""
    return 'model' in run.config and 'encoder' in run.config.get('model', {})

In [ ]:
# ----------------------------------------------------------------
# Check which parameter combinations already have completed runs
# ----------------------------------------------------------------
api = wandb.Api()
try:
    run_filters = {"group": WANDB_GROUP} if WANDB_GROUP else None
    existing_runs = api.runs(WANDB_PROJECT_PATH, filters=run_filters)
    msg = f"Found {len(existing_runs)} total runs in {WANDB_PROJECT_PATH}"
    if WANDB_GROUP:
        msg += f" (group={WANDB_GROUP})"
    print(msg)
except Exception as e:
    print(f"Could not query project (may not exist yet): {e}")
    existing_runs = []

finished_runs = [r for r in existing_runs if r.state == 'finished' and _is_jac_ode_run(r)]
already_done = []
remaining_combos = []
for combo in all_sweep_combos:
    if any(_combo_matches_run(combo, r) for r in finished_runs):
        already_done.append(combo)
        run_match = next(r for r in finished_runs if _combo_matches_run(combo, r))
        print(f"  Already done: {combo} (run_id={run_match.id})")
    else:
        remaining_combos.append(combo)

if already_done:
    print(f"\nSkipping {len(already_done)} already-completed combinations")
if remaining_combos:
    print(f"Need to run {len(remaining_combos)} combinations")
else:
    print("\nAll parameter combinations already have finished runs -- no sweep needed!")

In [ ]:
# ----------------------------------------------------------------
# Build the Hydra --multirun sweep command(s)
# ----------------------------------------------------------------
ENTRY_POINT = (
    "python -m JacobianODE.jacobians.run_jacobians"
    if MODE == "from_scratch"
    else "python -m JacobianODE.jacobians.run_pretrained_jacobians"
)

if remaining_combos:
    # Base overrides: exclude keys that are being swept
    _swept_keys = set(SWEEP_PARAMS.keys())
    def _is_swept(ov):
        for k in _swept_keys:
            if ov.strip().startswith(k + "="):
                return True
        return False
    fixed_overrides = [ov for ov in overrides if not _is_swept(ov)]

    n_remaining = len(remaining_combos)
    use_subset_sweep = n_remaining < n_sweep_combos

    if use_subset_sweep:
        # One command per remaining combo
        sweep_cmds = []
        for combo in remaining_combos:
            combo_overrides = [
                f"{key}={str(val).lower() if isinstance(val, bool) else val}"
                for key, val in combo.items()
            ]
            sweep_overrides = fixed_overrides + combo_overrides + [
                f"wandb_entity={WANDB_ENTITY}",
                f"wandb_project={WANDB_PROJECT}",
                "slurm=default",
            ]
            if WANDB_GROUP:
                sweep_overrides.append(f"wandb_group={WANDB_GROUP}")
            sweep_cmds.append(f"{ENTRY_POINT} --multirun " + " ".join(sweep_overrides))
    else:
        # Full grid sweep: comma-separated values
        sweep_param_overrides = []
        for key, vals in SWEEP_PARAMS.items():
            val_str = ",".join(str(v).lower() if isinstance(v, bool) else str(v) for v in vals)
            sweep_param_overrides.append(f"{key}={val_str}")
        sweep_overrides = fixed_overrides + sweep_param_overrides + [
            f"wandb_entity={WANDB_ENTITY}",
            f"wandb_project={WANDB_PROJECT}",
            "slurm=default",
        ]
        if WANDB_GROUP:
            sweep_overrides.append(f"wandb_group={WANDB_GROUP}")
        sweep_cmds = [f"{ENTRY_POINT} --multirun " + " ".join(sweep_overrides)]

    sweep_type = "subset (1 cmd per combo)" if use_subset_sweep else "grid"
    print(f"Sweep: {n_remaining} jobs ({sweep_type})")
    print(f"\nFirst command:\n{sweep_cmds[0][:200]}...")
else:
    sweep_cmds = []
    print("No sweep to launch -- all runs already completed.")

In [ ]:
# Launch the sweep (submits SLURM jobs)
if sweep_cmds:
    if len(sweep_cmds) > 1:
        print(f"Starting {len(sweep_cmds)} commands in parallel...")
    procs = [subprocess.Popen(cmd, shell=True) for cmd in sweep_cmds]
    for i, p in enumerate(procs):
        p.wait()
        if p.returncode != 0 and len(procs) > 1:
            print(f"Command {i + 1}/{len(procs)} exited with code {p.returncode}")
    print("Sweep complete.")
else:
    print("No sweep to launch -- all runs already completed.")

## Next Steps

Once all SLURM jobs have finished, use the companion
**Sweep Analytics (Latent JacobianODE)** notebook to:

1. Collect results from W&B
2. Apply physics-informed model selection (C1/C2/C3)
3. Load and diagnose the best model

Set the same `WANDB_PROJECT` and `WANDB_GROUP` in that notebook.